#Storytelling with Data: การเดินทางของประชาชนด้วยระบบขนส่งสาธารณะ

### ขั้นตอนการดำเนินงาน: นำเข้าข้อมูล -> ตรวจสอบเบื้องต้น -> ทำความสะอาดข้อมูล -> วิเคราะห์เชิงลึกหา Insight -> สรุปผลผ่าน Dashboard

Link Dataset: https://www.thackle.or.th/th/dataset/94

Link Colab: https://colab.research.google.com/drive/1o1AxacraBe9DgET0N57D_6O8DD37Ef_j?usp=sharing

##1.Setup & Load Data

###Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import find_peaks

# ตั้งค่าการแสดงผลเบื้องต้น
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'

### การโหลดข้อมูล (Load Data)

ในส่วนนี้ เราจะทำการนำเข้าชุดข้อมูลดิบของผู้โดยสารในปี พ.ศ. 2568 และ 2569 จาก Google Drive โดยจะทำการตรวจสอบโครงสร้างข้อมูลเบื้องต้นเพื่อให้เข้าใจลักษณะของตารางก่อนเริ่มดำเนินการ

In [ ]:
# โหลดข้อมูลปี 2568 และ 2569
df68 = pd.read_csv('/content/drive/MyDrive/SuperAI/Week1 - Data5/passengers68.csv')
df69 = pd.read_csv('/content/drive/MyDrive/SuperAI/Week1 - Data5/passengers69.csv')

print("df68 head:")
display(df68.head())
print("\ndf69 head:")
display(df69.head())

df68 head:


/tmp/ipykernel_6832/213735578.py:2: DtypeWarning:

Columns (0,1,2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.



,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ
0,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,บขส.,รถ บขส. และ รถร่วม,01/01/2025,คน,"127,551"
1,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,ขบ.,รถหมวด 3,01/01/2025,คน,"8,218"
2,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ),01/01/2025,คัน,"877,943"
3,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์ทุกประเภท (10 จุดสำรวจ),01/01/2025,คัน,"932,642"
4,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,กทพ.,รถยนต์เฉพาะ 4 ล้อ (ทางด่วน),01/01/2025,คัน,"1,364,992"



df69 head:


,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ
0,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,บขส.,รถ บขส. และ รถร่วม,01/01/2026,คน,"112,325"
1,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,ขบ.,รถหมวด 3,01/01/2026,คน,0
2,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ),01/01/2026,คัน,"892,218"
3,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์ทุกประเภท (10 จุดสำรวจ),01/01/2026,คัน,"980,649"
4,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,กทพ.,รถยนต์เฉพาะ 4 ล้อ (ทางด่วน),01/01/2026,คัน,"1,231,605"


## 2.First Look

### ภาพรวมและความไม่สมบูรณ์ของข้อมูล (Overall & Invalidity)

เราจะเริ่มจากการตรวจสอบคุณภาพข้อมูล (Data Quality Audit) เพื่อหาจุดบกพร่องก่อนการทำความสะอาด:
* **แถวที่ว่างเปล่า:** ตรวจสอบว่ามีแถวที่ไม่มีข้อมูลเลย (Null ทุกคอลัมน์) หรือไม่
* **คุณภาพคอลัมน์ปริมาณ:** ตรวจสอบชนิดข้อมูลและรูปแบบตัวเลข (เช่น การมีคอมม่าคั่น) ที่อาจเป็นอุปสรรคต่อการคำนวณ
* **ความสอดคล้องของหน่วย:** ตรวจสอบการใช้หน่วยวัด (เช่น 'คน' สำหรับผู้โดยสาร และ 'คัน' สำหรับยานพาหนะ) รวมถึงค่าที่สูญหายในหมวดหมู่ต่างๆ

In [ ]:
print('--- ตรวจสอบแถวว่าง (Completely Empty Rows) ---')
for name, df in [('df68', df68), ('df69', df69)]:
    ghost_rows = df.isnull().all(axis=1).sum()
    print(f"{name}: พบแถวว่างทั้งหมด {ghost_rows} แถว")

--- ตรวจสอบแถวว่าง (Completely Empty Rows) ---
df68: พบแถวว่างทั้งหมด 53744 แถว
df69: พบแถวว่างทั้งหมด 0 แถว


In [ ]:
print('--- ตรวจสอบคุณภาพข้อมูลคอลัมน์ ปริมาณ ---')
for name, df in [('df68', df68), ('df69', df69)]:
    nans = df['ปริมาณ'].isnull().sum()
    dtype = df['ปริมาณ'].dtype
    commas = df['ปริมาณ'].dropna().astype(str).str.contains(',').sum() if dtype == 'object' else 0
    print(f">>> {name}: ชนิดข้อมูล={dtype}, ค่าว่าง={nans}, แถวที่มีคอมมา={commas}")

--- ตรวจสอบคุณภาพข้อมูลคอลัมน์ ปริมาณ ---
>>> df68: ชนิดข้อมูล=object, ค่าว่าง=54052, แถวที่มีคอมมา=12692
>>> df69: ชนิดข้อมูล=object, ค่าว่าง=136, แถวที่มีคอมมา=2377


In [ ]:
## Invalid values or blank rows
print('--- Invalid 3 ---')
for name, df in [('df68', df68), ('df69', df69)]:
    unit_counts = df['หน่วย'].value_counts(dropna=False)
    print(f'Unit distribution in {name}:')
    print(unit_counts)
    print('-' * 20)

--- Invalid 3 ---
Unit distribution in df68:
หน่วย
NaN    53744
คน     14236
คัน     1460
Name: count, dtype: int64
--------------------
Unit distribution in df69:
หน่วย
คน     2730
คัน     280
Name: count, dtype: int64
--------------------


###Visualization

#### 1. ภาพรวมกระบวนการสร้างภาพข้อมูล (Visualization Process)
เปลี่ยนจากการตรวจสอบด้วยตัวเลขเป็นการใช้กราฟเพื่อให้เห็นปัญหาชัดเจนยิ่งขึ้น โดยเน้น 3 ประเด็นหลัก:
* **การตรวจพบแถวว่าง (Ghost Rows):** แสดงสัดส่วนของข้อมูลที่ใช้ได้เทียบกับแถวที่ว่างเปล่า
* **การวิเคราะห์ค่าปริมาณที่หายไป:** ดูว่าการเดินทางประเภทใดที่ขาดข้อมูล 'ปริมาณ' มากที่สุด
* **การกระจายตัวของหน่วย:** ตรวจสอบความสอดคล้องของหน่วยวัดระหว่างปี 2568 และ 2569

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Prep Data
stats = []
for name, d in [('df68', df68), ('df69', df69)]:
    total = len(d)
    ghost = d.isnull().all(axis=1).sum()
    stats.append({'name': name, 'valid': total-ghost, 'ghost': ghost})

# Using vertical_spacing to ensure titles don't overlap labels
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type':'pie'}, {'type':'pie'}]],
    subplot_titles=['Year 2568', 'Year 2569']
)

fig.add_trace(go.Pie(labels=['Valid', 'Invalid'], values=[stats[0]['valid'], stats[0]['ghost']], name='df68', hole=0.3), 1, 1)
fig.add_trace(go.Pie(labels=['Valid', 'Invalid'], values=[stats[1]['valid'], stats[1]['ghost']], name='df69', hole=0.3), 1, 2)

# Adjust layout: 't' for top margin, and move titles slightly up
fig.update_layout(
    title_text='Comparison: Valid vs. Invalid Data Rows',
    title_x=0.5,
    margin=dict(t=150, b=50, l=50, r=50),
    showlegend=True
)

# Update subplot titles font size and position if needed
for i in range(len(fig.layout.annotations)):
    fig.layout.annotations[i].font.size = 18
    fig.layout.annotations[i].y += 0.05

fig.show()

#### 2. บทวิเคราะห์จากการสำรวจข้อมูลเบื้องต้น (Basic EDA)
จากกราฟที่ปรากฏ เราพบประเด็นสำคัญดังนี้:
* **ปัญหา Ghost Rows:** ข้อมูลปี 2568 มีแถวว่างสูงถึงเกือบ 77% ในขณะที่ปี 2569 มีความสะอาดกว่ามาก ทำให้เราต้องคัดกรองข้อมูลปี 2568 อย่างเข้มงวดก่อนนำไปใช้
* **รูปแบบของค่าที่สูญหาย:** ค่าปริมาณมักหายไปในหมวดหมู่ 'ส่วนบุคคล' ซึ่งนำไปสู่สมมติฐานว่า *ระบบการจัดเก็บข้อมูลการเดินทางส่วนบุคคลอาจมีความแม่นยำน้อยกว่าการขนส่งสาธารณะ*
* **หน่วยวัดที่สอดคล้อง:** ทั้งสองปีใช้หน่วย 'คน' เป็นหลักสำหรับการขนส่งสาธารณะ ทำให้เราสามารถเปรียบเทียบอัตราการเติบโตได้อย่างแม่นยำ

In [ ]:
# Calculate missing % relative to total rows in each dataset
results = []
for name, df in [('df68', df68), ('df69', df69)]:
    total_valid_rows = len(df.dropna(how='all'))
    temp = df[df['ปริมาณ'].isnull()]['สาธารณะ/ส่วนบุคคล'].value_counts().reset_index()
    temp.columns = ['Category', 'Count']
    temp['Percentage'] = (temp['Count'] / total_valid_rows) * 100
    temp['Year'] = name
    results.append(temp)

combined_missing = pd.concat(results)

fig = px.bar(combined_missing, x='Percentage', y='Category', color='Year', barmode='group',
             orientation='h', title='Missing Volume as % of Total Yearly Data', text_auto='.2f')
fig.update_layout(xaxis_title='Percentage of Total Data (%)')
fig.show()

In [ ]:
def get_unit_pct_of_total(df, name):
    valid_df = df.dropna(how='all')
    total_rows = len(valid_df)
    temp = valid_df.groupby(['สาธารณะ/ส่วนบุคคล', 'หน่วย']).size().reset_index(name='count')
    # Percentage based on total rows in the entire dataset for that year
    temp['Percentage'] = (temp['count'] / total_rows) * 100
    temp['Year'] = name
    return temp

combined_unit = pd.concat([get_unit_pct_of_total(df68, 'df68'), get_unit_pct_of_total(df69, 'df69')])

fig = px.bar(combined_unit, x='สาธารณะ/ส่วนบุคคล', y='Percentage', color='หน่วย', facet_col='Year',
             title='Unit Distribution as % of Total Yearly Data', barmode='stack', text_auto='.1f')
fig.update_layout(yaxis_title='Percentage of Total Records (%)')
fig.show()

#### 3. เหตุผลในการเลือกใช้กราฟ
*   **กราฟวงกลม (Pie Chart):** เหมาะสำหรับการแสดงสัดส่วน 'ดี vs เสีย' ทำให้เห็นขนาดของปัญหาข้อมูลเน่าเสียในปี 2568 ได้ทันที
*   **กราฟแท่งแนวนอนแบบกลุ่ม (Grouped Bar Chart):** ช่วยให้เปรียบเทียบค่าที่สูญหายระหว่าง 2 ปีในแต่ละหมวดหมู่ได้อย่างชัดเจน
*   **กราฟแท่งแบบซ้อน (Stacked Bar Chart):** ใช้แสดงองค์ประกอบภายในของหน่วยวัดในแต่ละประเภทการเดินทาง โดยยังรักษามาตรวัดที่เปรียบเทียบกันได้

## 3.Data Cleaning

### การทำความสะอาดเบื้องต้น (Basic Cleaning)

ในขั้นตอนนี้ เราจัดการกับปัญหาคุณภาพข้อมูลที่วิกฤตที่สุด:
*   **กำจัดแถวว่าง (Ghost Rows):** ตัดแถวที่ไม่มีข้อมูลในทุกคอลัมน์ออก
*   **ตรวจสอบค่าปริมาณที่หายไป:** คัดกรองและแสดงผลแถวที่ข้อมูล 'ปริมาณ' เป็นค่าว่างเพื่อทำความเข้าใจก่อนตัดสินใจลบหรือแทนที่ค่าเหล่านั้น
*   **แปลงชนิดข้อมูล:** ปรับแต่งคอลัมน์ 'ปริมาณ' โดยลบคอมม่าและแปลงเป็นตัวเลขเพื่อใช้ในการคำนวณ

### Basic Cleaning

In [ ]:
# ลบแถวที่เป็นค่าว่างทั้งหมด
df68 = df68.dropna(how='all')
df69 = df69.dropna(how='all')

print(f"df68: {df68.shape}")
print(f"df69: {df69.shape}")

df68: (15696, 8)
df69: (3010, 8)


In [ ]:
# Use .isna().sum() to see a quick count of blanks per column
print("--- Quick Check for Missing Volumes ---")
print(df68.isna().sum())

# Filter to get the exact rows where the 'ปริมาณ' (Volume) is blank
missing_value_rows = df68[df68['ปริมาณ'].isna()]

# VISUALIZE VIA TABLE (Using Plotly)
sample_missing = missing_value_rows.head(10)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(sample_missing.columns), # Use column names as headers
        fill_color='navy',
        font=dict(color='white', size=14),
        align='left'
    ),
    cells=dict(
        # Plotly Tables require data in a list-of-lists format (one list per column)
        values=[sample_missing[col] for col in sample_missing.columns],
        fill_color='aliceblue',
        font=dict(color='black', size=12),
        align='left'
    )
)])

fig.update_layout(
    title='ตัวอย่างตารางข้อมูลที่สูญหาย (Sample of Rows with Blank Volumes)',
    margin=dict(l=20, r=20, t=40, b=20)
)

fig.show()

--- Quick Check for Missing Volumes ---
รูปแบบการเดินทาง       0
วัตถุประสงค์           0
สาธารณะ/ส่วนบุคคล      0
หน่วยงาน               0
ยานพาหนะ/ท่า           0
วันที่                 0
หน่วย                  0
ปริมาณ               308
dtype: int64


In [ ]:
## Drop rows with missing volume using .copy() to avoid SettingWithCopyWarning
df68_clean = df68.dropna(subset=['ปริมาณ']).copy()

In [ ]:
# ลบค่าว่างในคอลัมน์ปริมาณ และแปลงเป็นตัวเลข
df68_clean = df68.dropna(subset=['ปริมาณ']).copy()

for df in [df68_clean, df69]:
    df['ปริมาณ'] = df['ปริมาณ'].astype(str).str.replace(',', '')
    df['ปริมาณ'] = pd.to_numeric(df['ปริมาณ'], errors='coerce')

print(f"Data types: df68_clean={df68_clean['ปริมาณ'].dtype}, df69={df69['ปริมาณ'].dtype}")

Data types: df68_clean=int64, df69=float64


### การทำความสะอาดเชิงลึก: ข้อมูลซ้ำ, วันที่ และค่าที่ผิดปกติ

เพื่อให้ข้อมูลอนุกรมเวลา (Time-series) มีความถูกต้องตามตรรกะ:
*   **การจัดการข้อมูลซ้ำ:** ตรวจสอบและลบแถวที่มีข้อมูลเหมือนกันทุกประการเพื่อป้องกันการนับซ้ำ (Overcounting)
*   **การปรับมาตรฐานวันที่:** แปลงคอลัมน์ 'วันที่' ให้เป็นรูปแบบ Datetime ที่ถูกต้อง เพื่อใช้ในการเรียงลำดับและวิเคราะห์แนวโน้ม
*   **การตรวจสอบความถูกต้อง:** ยืนยันว่าไม่มีค่าปริมาณที่เป็นลบ ซึ่งเป็นไปไม่ได้ในข้อมูลการเดินทางจริง

In [ ]:
# Check for duplicates
for name, df in [('df68_clean', df68_clean), ('df69', df69)]:
    dup_count = df.duplicated().sum()
    print(f'{name} has {dup_count} duplicated rows.')

print("-"*20)

# Check for negative values in Volume
for name, df in [('df68_clean', df68_clean), ('df69', df69)]:
    neg_count = (df['ปริมาณ'] < 0).sum()
    print(f'{name} has {neg_count} negative volume values.')

# Preview date formats
print('\nSample dates from df68_clean:', df68_clean['วันที่'].head(3).tolist())
print('Sample dates from df69:', df69['วันที่'].head(3).tolist())

df68_clean has 1 duplicated rows.
df69 has 0 duplicated rows.
--------------------
df68_clean has 0 negative volume values.
df69 has 0 negative volume values.

Sample dates from df68_clean: ['01/01/2025', '01/01/2025', '01/01/2025']
Sample dates from df69: ['01/01/2026', '01/01/2026', '01/01/2026']


In [ ]:
# Identify and display duplicated rows in df68_clean
duplicates = df68_clean[df68_clean.duplicated(keep=False)]

if not duplicates.empty:
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(duplicates.columns), fill_color='darkred', font=dict(color='white')),
        cells=dict(values=[duplicates[col] for col in duplicates.columns], fill_color='lavender')
    )])
    # Set height based on row count and tighten margins to remove blank space
    fig.update_layout(
        title='Duplicated Rows Found in df68_clean',
        height=250,
        margin=dict(l=10, r=10, t=60, b=10)
    )
    fig.show()
else:
    print('No duplicates found in the current state of df68_clean.')

In [ ]:
# Remove duplicated rows from df68_clean
initial_rows = len(df68_clean)
df68_clean = df68_clean.drop_duplicates().copy()
final_rows = len(df68_clean)

print(f'Rows before cleaning: {initial_rows}')
print(f'Rows after cleaning: {final_rows}')
print(f'Successfully removed {initial_rows - final_rows} duplicate(s).')

Rows before cleaning: 15388
Rows after cleaning: 15387
Successfully removed 1 duplicate(s).


In [ ]:
# แปลงคอลัมน์วันที่เป็น datetime format
for df in [df68_clean, df69]:
    df['วันที่'] = pd.to_datetime(df['วันที่'], format='%d/%m/%Y')

print("การแปลงวันที่เสร็จสมบูรณ์")
display(df68_clean['วันที่'].head(3))

การแปลงวันที่เสร็จสมบูรณ์


,วันที่
0,2025-01-01
1,2025-01-01
2,2025-01-01


### Additional stripping

In [ ]:
# Clean all string columns by stripping leading/trailing whitespace
# We keep this to ensure data consistency even if the source data changes
for df in [df68_clean, df69]:
    str_cols = df.select_dtypes(include=['object']).columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()

print('Result: All categorical columns have been sanitized (whitespace stripped).')

Result: All categorical columns have been sanitized (whitespace stripped).


## 4.Advanced EDA & Insights

### Challenge 1: การวิเคราะห์ส่วนแบ่งตลาดและอัตราการเติบโตของระบบราง

ในส่วนนี้ เราจะเจาะลึกว่าระบบรถไฟฟ้าเครือข่ายใดครองส่วนแบ่งการตลาดสูงสุด และสำรวจแนวโน้มการเติบโตระหว่างปี 2568 และ 2569

**แนวทางการวิเคราะห์:**
1.  **การจัดกลุ่ม:** รวมสายสีต่างๆ (น้ำเงิน, ม่วง, เหลือง, ชมพู) เข้าด้วยกันภายใต้การบริหารของ MRT และแยกวิเคราะห์ BTS, ARL และสายสีแดง
2.  **ส่วนแบ่งตลาด:** ใช้ Stacked Area Chart เพื่อดูสัดส่วนการครองตลาดรายวัน
3.  **การเปรียบเทียบการเติบโต:** ใช้ช่วงเวลาเดียวกัน (YTD) คือ 1 ม.ค. - 11 มี.ค. ของทั้งสองปีเพื่อการเปรียบเทียบที่แม่นยำและไม่ลำเอียง

### Mapping rails to groups

In [ ]:
# รวมข้อมูลและกรองเฉพาะ 'ทางราง'
df_all = pd.concat([df68_clean, df69], ignore_index=True)
df_rail = df_all[df_all['รูปแบบการเดินทาง'] == 'ทางราง'].copy()

# จัดกลุ่มเครือข่ายรถไฟฟ้า
rail_map = {
    'รถไฟฟ้า BTS': 'BTS',
    'รถไฟฟ้าสายสีน้ำเงิน': 'MRT', 'รถไฟฟ้าสายสีม่วง': 'MRT',
    'รถไฟฟ้าสายสีเหลือง': 'MRT', 'รถไฟฟ้าสายสีชมพู': 'MRT',
    'รถไฟฟ้า ARL': 'Airport Rail Link',
    'รถไฟฟ้าสายสีแดง': 'SRT Red Line'
}

df_rail['กลุ่มรถไฟฟ้า'] = df_rail['ยานพาหนะ/ท่า'].map(rail_map)
df_rail = df_rail.dropna(subset=['กลุ่มรถไฟฟ้า'])

display(df_rail.groupby('กลุ่มรถไฟฟ้า')['ปริมาณ'].sum().reset_index())

,กลุ่มรถไฟฟ้า,ปริมาณ
0,Airport Rail Link,28680717.0
1,BTS,314037449.0
2,MRT,261651648.0
3,SRT Red Line,15862537.0


### Challenge  Question 1

In [ ]:
# Challenge 1: Market Share Analysis (Area Chart)
df_pivot = df_rail.pivot_table(index='วันที่', columns='กลุ่มรถไฟฟ้า', values='ปริมาณ', aggfunc='sum', fill_value=0)
df_perc = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100

fig = px.area(df_perc.reset_index(), x='วันที่', y=['BTS', 'MRT', 'Airport Rail Link', 'SRT Red Line'],
              title='<b>Market Share ของระบบรถไฟฟ้า (2568 - 2569)</b>',
              labels={'value': 'สัดส่วน (%)', 'variable': 'เครือข่าย'},
              color_discrete_map={'BTS':'#42b944', 'MRT':'#183884', 'SRT Red Line':'#d3273e', 'Airport Rail Link':'#8a2132'})

fig.update_layout(plot_bgcolor='white', hovermode='x unified')
fig.show()

In [ ]:
# 1. Ensure date types are correct and create time features
df_rail['วันที่'] = pd.to_datetime(df_rail['วันที่'])
df_rail['เดือน'] = df_rail['วันที่'].dt.month
df_rail['วัน'] = df_rail['วันที่'].dt.day
df_rail['ปี'] = df_rail['วันที่'].dt.year

# 2. Define YTD Mask (Jan 1st - Mar 11th) to ensure like-for-like comparison
ytd_mask = (df_rail['เดือน'] < 3) | ((df_rail['เดือน'] == 3) & (df_rail['วัน'] <= 11))
df_ytd = df_rail[ytd_mask].copy()

# 3. Calculate Daily Averages per Year per Group
# First sum up by day (in case of multiple entries per day)
daily_sum = df_ytd.groupby(['ปี', 'วันที่', 'กลุ่มรถไฟฟ้า'])['ปริมาณ'].sum().reset_index()
# Then calculate the mean of those daily sums per year
yearly_avg = daily_sum.groupby(['ปี', 'กลุ่มรถไฟฟ้า'])['ปริมาณ'].mean().reset_index()

# 4. Pivot and Calculate Growth
df_pivot = yearly_avg.pivot(index='กลุ่มรถไฟฟ้า', columns='ปี', values='ปริมาณ').reset_index()
df_pivot['Growth (%)'] = ((df_pivot[2026] - df_pivot[2025]) / df_pivot[2025]) * 100
df_pivot = df_pivot.sort_values('Growth (%)', ascending=True)

# 5. Create Visualization
fig2 = go.Figure()

colors = []
for _, row in df_pivot.iterrows():
    if row['Growth (%)'] > 0:
        colors.append('#2ca02c') # Green for growth
    else:
        colors.append('#d62728') # Red for decline

fig2.add_trace(go.Bar(
    x=df_pivot['Growth (%)'],
    y=df_pivot['กลุ่มรถไฟฟ้า'],
    orientation='h',
    marker_color=colors,
    text=df_pivot['Growth (%)'].apply(lambda x: f"{x:+.2f}%"),
    textposition='outside'
))

fig2.update_layout(
    title='<b>การตรวจสอบอัตราการเติบโต (YTD: 1 ม.ค. - 11 มี.ค.)</b><br><sup>เปรียบเทียบค่าเฉลี่ยรายวัน 2568 vs 2569</sup>',
    xaxis_title='Growth (%)',
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='LightGray', zeroline=True, zerolinecolor='black'),
    margin=dict(l=20, r=60, t=80, b=40)
)

fig2.show()

#### บทวิเคราะห์ Challenge 1: การครองตลาดและการเติบโตของระบบราง

**1. พลวัตส่วนแบ่งตลาด (Area Chart)**
*   **ทำไมต้องกราฟนี้?** เราใช้ **Stacked Area Chart** (แบบ Normalized 100%) เพื่อจำลอง 'อาณาเขต' ของแต่ละเครือข่ายตามช่วงเวลา ซึ่งทำให้เห็นความยิ่งใหญ่ที่สม่ำเสมอของ BTS และ MRT ในขณะที่ยังสังเกตเห็นการขยับตัวของเครือข่ายขนาดเล็กอย่าง ARL และสายสีแดงได้ชัดเจน
*   **ข้อมูลเชิงลึก:** BTS ยังคงเป็นกระดูกสันหลังหลัก ตามมาด้วย MRT อย่างใกล้ชิด ส่วนสายสีแดงและ ARL แม้จะมีส่วนแบ่งน้อยกว่าแต่มีฐานผู้โดยสารที่คงที่ สะท้อนถึงกลุ่มผู้ใช้เฉพาะเส้นทางที่เหนียวแน่น

**2. ประสิทธิภาพการเติบโต (Horizontal Bar Chart)**
*   **ทำไมต้องกราฟนี้?** **กราฟแท่งแนวนอน** พร้อมรหัสสี (เขียวเพื่อการเติบโต, แดงเพื่อการหดตัว) ช่วยให้จัดอันดับประสิทธิภาพได้ทันที โดยการเน้นที่ค่าเฉลี่ยรายวันในช่วง 1 ม.ค. - 11 มี.ค. ของทั้งสองปี ทำให้เป็นการเปรียบเทียบที่ยุติธรรมแบบ 'เทียบสัดส่วนจริง'
*   **ข้อมูลเชิงลึก:** กราฟนี้เผยให้เห็นว่าผู้ให้บริการรายใดขยายฐานลูกค้าได้สำเร็จ การเติบโตสูงในสายสีแดงหรือ ARL อาจบ่งชี้ถึงการยอมรับระบบ Feeder มากขึ้น หรือการฟื้นตัวของการเดินทางระหว่างประเทศ ในขณะที่การเปลี่ยนแปลงของ BTS/MRT สะท้อนถึงสุขภาพการเดินทางในใจกลางเมือง

### Challenge Question 2

### Challenge 2: การเปรียบเทียบรถไฟฟ้าในเมือง — ความผันผวนและพฤติกรรมผู้โดยสาร

เราจะวิเคราะห์เจาะลึกรายเส้นทางเพื่อทำความเข้าใจลักษณะเฉพาะของผู้โดยสาร เนื่องจากรถไฟฟ้าแต่ละสายทำหน้าที่ต่างกัน ทั้งสายหลัก (Core), สายรอง (Feeder) หรือสายเชื่อมต่อสนามบิน

**แนวทางการวิเคราะห์:**
1.  **การเปรียบเทียบรายสาย:** วิเคราะห์ 7 เส้นทางหลักอย่างอิสระ
2.  **การกระจายตัว:** ใช้ Box Plot เพื่อดูช่วงการเดินทางปกติ รวมถึงจุดพีกและจุดต่ำสุดในแต่ละวัน
3.  **การประเมินความผันผวน:** คำนวณค่า CV% และ Weekend Drop % เพื่อจำแนกความเสถียรของฐานผู้โดยสาร

In [ ]:
# 1. Data Preparation
target_lines = [
    'รถไฟฟ้า BTS', 'รถไฟฟ้าสายสีน้ำเงิน', 'รถไฟฟ้าสายสีม่วง',
    'รถไฟฟ้าสายสีเหลือง', 'รถไฟฟ้าสายสีชมพู', 'รถไฟฟ้า ARL', 'รถไฟฟ้าสายสีแดง'
]
df_challenge2 = df_rail[df_rail['ยานพาหนะ/ท่า'].isin(target_lines)].copy()

rename_dict = {
    'รถไฟฟ้า BTS': '1. BTS',
    'รถไฟฟ้าสายสีน้ำเงิน': '2. MRT Blue',
    'รถไฟฟ้าสายสีม่วง': '3. MRT Purple',
    'รถไฟฟ้าสายสีเหลือง': '4. MRT Yellow',
    'รถไฟฟ้าสายสีชมพู': '5. MRT Pink',
    'รถไฟฟ้า ARL': '6. ARL',
    'รถไฟฟ้าสายสีแดง': '7. SRT Red'
}
df_challenge2['สายรถไฟฟ้า'] = df_challenge2['ยานพาหนะ/ท่า'].map(rename_dict)

# Color mapping based on rail line brand identity
color_map = {
    '1. BTS': '#42b944',      # BTS Green
    '2. MRT Blue': '#183884', # MRT Blue
    '3. MRT Purple': '#800080',# MRT Purple
    '4. MRT Yellow': '#ffcc00',# MRT Yellow
    '5. MRT Pink': '#ff66cc',  # MRT Pink
    '6. ARL': '#8a2132',       # ARL Maroon
    '7. SRT Red': '#d3273e'    # SRT Red
}

# 2. Split into Tiers
tier1_names = ['1. BTS', '2. MRT Blue']
df_tier1 = df_challenge2[df_challenge2['สายรถไฟฟ้า'].isin(tier1_names)]
df_tier2 = df_challenge2[~df_challenge2['สายรถไฟฟ้า'].isin(tier1_names)]

# 3. Create Subplots
fig3 = make_subplots(
    rows=2, cols=1,
    subplot_titles=('<b>กลุ่มที่ 1: เส้นเลือดใหญ่ (Core Network)</b>',
                    '<b>กลุ่มที่ 2: เส้นเลือดฝอย (Feeder Network)</b>'),
    vertical_spacing=0.15
)

# 4. Add Box Plots for Tier 1
for line in sorted(df_tier1['สายรถไฟฟ้า'].unique()):
    subset = df_tier1[df_tier1['สายรถไฟฟ้า'] == line]
    fig3.add_trace(go.Box(y=subset['ปริมาณ'], name=line, boxpoints='all',
                          marker_color=color_map.get(line), jitter=0.3, pointpos=-1.8), row=1, col=1)

# 5. Add Box Plots for Tier 2
for line in sorted(df_tier2['สายรถไฟฟ้า'].unique()):
    subset = df_tier2[df_tier2['สายรถไฟฟ้า'] == line]
    fig3.add_trace(go.Box(y=subset['ปริมาณ'], name=line, boxpoints='all',
                          marker_color=color_map.get(line), jitter=0.3, pointpos=-1.8), row=2, col=1)

# 6. Layout Styling
fig3.update_layout(
    height=850,
    showlegend=False,
    title_text='<b>Challenge 2: การกระจายตัวและความผันผวนของผู้โดยสาร (จำแนกตามสีสายรถไฟฟ้า)</b>',
    plot_bgcolor='white'
)

fig3.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', row=1, col=1)
# Adding zeroline for the feeder network (Row 2)
fig3.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', zeroline=True, zerolinecolor='black', zerolinewidth=1, row=2, col=1)

fig3.show()

In [ ]:
# 1. ปรับชื่อสายรถไฟฟ้าเพื่อลบตัวเลขนำหน้า (Clean Names)
clean_rename = {
    '1. BTS': 'BTS',
    '2. MRT Blue': 'MRT Blue',
    '3. MRT Purple': 'MRT Purple',
    '4. MRT Yellow': 'MRT Yellow',
    '5. MRT Pink': 'MRT Pink',
    '6. ARL': 'ARL',
    '7. SRT Red': 'SRT Red'
}
df_challenge2['สายรถไฟฟ้า'] = df_challenge2['สายรถไฟฟ้า'].replace(clean_rename)

# 2. คำนวณความผันผวน (CV) และปริมาณลดลงในวันหยุด (Weekend Drop)
stats = df_challenge2.groupby('สายรถไฟฟ้า')['ปริมาณ'].agg(['mean', 'std']).reset_index()
stats['CV (%)'] = (stats['std'] / stats['mean']) * 100

df_challenge2['Is_Weekend'] = df_challenge2['วันที่'].dt.dayofweek.isin([5, 6])
agg_drop = df_challenge2.groupby(['สายรถไฟฟ้า', 'Is_Weekend'])['ปริมาณ'].mean().unstack()
agg_drop.columns = ['Weekday_Avg', 'Weekend_Avg']
agg_drop['Weekend_Drop (%)'] = ((agg_drop['Weekday_Avg'] - agg_drop['Weekend_Avg']) / agg_drop['Weekday_Avg']) * 100

# รวมตารางและเรียงลำดับตาม CV จากน้อยไปมาก (เสถียรอยู่บน ผันผวนอยู่ล่าง)
stats = pd.merge(stats, agg_drop.reset_index(), on='สายรถไฟฟ้า').sort_values('CV (%)', ascending=True)

# 3. สร้างโครงกราฟแบบ 1 แถว 2 คอลัมน์ (Side-by-Side)
fig_summary = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>ความผันผวนโดยรวม (CV %)</b><br><sup>ยิ่งเยอะ = ยิ่งสวิงแรง</sup>',
                    '<b>การหดตัวในวันหยุด (Weekend Drop %)</b><br><sup>ผู้โดยสารหายไปกี่ % ในวันเสาร์-อาทิตย์</sup>'),
    horizontal_spacing=0.15
)

# 4. ฟังก์ชันกำหนดสีตามเกณฑ์ความเสี่ยง
def get_colors(values, threshold_high, threshold_low):
    return ['#cc0000' if v >= threshold_high else '#e67e22' if v >= threshold_low else '#bdc3c7' for v in values]

cv_colors = get_colors(stats['CV (%)'], 26, 22)
drop_colors = get_colors(stats['Weekend_Drop (%)'], 40, 30)

# 5. เพิ่มกราฟแท่ง (Horizontal Bars)
fig_summary.add_trace(go.Bar(
    x=stats['CV (%)'], y=stats['สายรถไฟฟ้า'], orientation='h', marker_color=cv_colors,
    text=stats['CV (%)'].apply(lambda x: f"{x:.1f}%"), textposition='outside', name='Volatility'
), row=1, col=1)

fig_summary.add_trace(go.Bar(
    x=stats['Weekend_Drop (%)'], y=stats['สายรถไฟฟ้า'], orientation='h', marker_color=drop_colors,
    text=stats['Weekend_Drop (%)'].apply(lambda x: f"{x:.1f}%"), textposition='outside', name='Weekend Drop'
), row=1, col=2)

# 6. ปรับแต่งเลย์เอาต์
fig_summary.update_layout(
    title='<b>สรุป: อันดับความผันผวนและเสถียรภาพของผู้โดยสารระบบราง</b>',
    plot_bgcolor='white', showlegend=False, height=500,
    margin=dict(t=100, b=40, l=20, r=20)
)

fig_summary.update_yaxes(showgrid=False, row=1, col=1)
fig_summary.update_yaxes(showgrid=False, showticklabels=False, row=1, col=2)
fig_summary.update_xaxes(showgrid=True, gridcolor='LightGray', range=[0, 35], row=1, col=1)
fig_summary.update_xaxes(showgrid=True, gridcolor='LightGray', range=[0, 50], row=1, col=2)

fig_summary.show()

**บทวิเคราะห์สรุปความผันผวนและเสถียรภาพ (Volatility Analysis)**

จากการวิเคราะห์ข้อมูลผู้โดยสารระบบรางผ่านตัวชี้วัด 2 รูปแบบหลัก คือ **Coefficient of Variation (CV %)** และ **Weekend Drop (%)** เราสามารถสรุปประเด็นสำคัญได้ดังนี้:

**1. ตรรกะการใช้สี (Color Coding Logic)**
เพื่อให้ผู้บริหารหรือผู้ใช้งานระบุความเสี่ยงได้ทันที เราจึงกำหนดเกณฑ์สีตามระดับความผันผวน:
*   🔴 **สีแดง (Critical):** มีความผันผวนสูงมาก (CV > 26% หรือ Weekend Drop > 40%) เช่น **MRT สายสีม่วง** และ **MRT สายสีชมพู** ข้อมูลกลุ่มนี้ต้องการการวางแผนทรัพยากรที่ยืดหยุ่นสูงในวันหยุด
*   🟠 **สีส้ม (Warning):** มีความผันผวนระดับปานกลาง (CV 22-26% หรือ Weekend Drop 30-40%) เช่น **MRT สายสีน้ำเงิน**
*   🔘 **สีเทา (Stable):** มีความเสถียรสูง (CV < 22% หรือ Weekend Drop < 30%) เช่น **Airport Rail Link (ARL)** และ **BTS** ซึ่งมีฐานผู้โดยสารสม่ำเสมอทั้งวันทำงานและวันหยุด

**2. รูปแบบฟันปลา (Sawtooth Pattern)**
กราฟเปรียบเทียบแสดงให้เห็นชัดเจนว่า:
*   **สายที่ผันผวนสูง (กลุ่ม Feeder):** อย่างสายสีม่วงและสายสีชมพู จะพบพฤติกรรม "วันหยุดผู้โดยสารหาย" ที่รุนแรง (Weekend Drop สูงถึง 40-45%) เนื่องจากผู้โดยสารส่วนใหญ่เป็นกลุ่มคนทำงาน (Commuters) ที่เดินทางเข้าเมือง
*   **สายที่เสถียร (กลุ่ม Core/Tourist):** เช่น ARL มี Weekend Drop เพียง 25% เนื่องจากเป็นสายที่เชื่อมต่อสนามบิน ซึ่งมีความต้องการเดินทางสูงตลอดทั้งสัปดาห์ ไม่จำกัดเฉพาะวันทำงาน

**3. ข้อเสนอแนะเชิงกลยุทธ์**
*   สำหรับกลุ่ม **สีแดง**, การลดความถี่ของขบวนรถในวันหยุดอาจช่วยลดต้นทุนการดำเนินงานได้
*   สำหรับกลุ่ม **สีเทา**, ควรคงระดับบริการให้สม่ำเสมอเนื่องจากความต้องการเดินทางมีความเสถียรตลอด 7 วัน

In [ ]:
# 1. Prepare clean names by removing numeric prefixes if they exist in df_challenge2
# This ensures consistency with the user's request across the visualization
clean_rename = {
    '1. BTS': 'BTS',
    '2. MRT Blue': 'MRT Blue',
    '3. MRT Purple': 'MRT Purple',
    '4. MRT Yellow': 'MRT Yellow',
    '5. MRT Pink': 'MRT Pink',
    '6. ARL': 'ARL',
    '7. SRT Red': 'SRT Red'
}
df_challenge2['สายรถไฟฟ้า'] = df_challenge2['สายรถไฟฟ้า'].replace(clean_rename)

# 2. ดึงข้อมูล 1 เดือนเต็ม (กุมภาพันธ์ 2568)
df_month = df_challenge2[(df_challenge2['วันที่'] >= '2025-02-01') & (df_challenge2['วันที่'] <= '2025-02-28')].copy()
df_month['Max_Volume'] = df_month.groupby('สายรถไฟฟ้า')['ปริมาณ'].transform('max')
df_month['% of Peak'] = (df_month['ปริมาณ'] / df_month['Max_Volume']) * 100

# 3. สร้าง Facet Grid
fig2 = px.line(
    df_month,
    x='วันที่',
    y='% of Peak',
    facet_col='สายรถไฟฟ้า',
    facet_col_wrap=4,
    color='สายรถไฟฟ้า',
    markers=True,
    title='<b>2. รูปแบบการเปลี่ยนแปลงของผู้โดยสารรายวัน (ตลอดเดือนกุมภาพันธ์ 2568)</b><br><sup>แสดงเทรนด์ยาว 4 สัปดาห์ เพื่อยืนยันพฤติกรรม "ฟันปลา" (Weekend Drop)</sup>',
    color_discrete_map={
        'BTS': '#42b944', 'MRT Blue': '#183884', 'MRT Purple': '#800080',
        'MRT Yellow': '#ffcc00', 'MRT Pink': '#ff66cc', 'ARL': '#8a2132', 'SRT Red': '#d3273e'
    }
)

# 4. ไฮไลต์วันหยุดสุดสัปดาห์ (เสาร์-อาทิตย์)
weekends_feb = [
    ("2025-02-01", "2025-02-02"), ("2025-02-08", "2025-02-09"),
    ("2025-02-15", "2025-02-16"), ("2025-02-22", "2025-02-23")
]
for start, end in weekends_feb:
    fig2.add_vrect(
        x0=start, x1=end, fillcolor="#f0f0f0", opacity=1, line_width=0, layer="below", row="all", col="all"
    )

# 5. ปรับแต่ง Layout
fig2.update_layout(
    plot_bgcolor='white',
    showlegend=False,
    margin=dict(t=150, b=50, l=50, r=50),
    height=600,
    hovermode='x unified'
)

# ปรับแกน X ให้แสดงวันที่ทุก 7 วัน
fig2.update_xaxes(
    showgrid=False,
    title='',
    tickvals=pd.date_range("2025-02-01", "2025-02-28", freq="7D"),
    tickformat="%d %b"
)

fig2.update_yaxes(showgrid=True, gridcolor='LightGray', ticksuffix="%", range=[35, 110])

# ลบส่วนเกินหัวข้อและแต่ง Tooltip
fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig2.update_traces(hovertemplate='<b>%{y:.1f}%</b><extra></extra>')

fig2.show()

**บทวิเคราะห์สรุป: รูปแบบการเปลี่ยนแปลงของผู้โดยสารรายวัน (Daily Passenger Pattern Analysis)**

จากการวิเคราะห์กราฟแนวโน้มรายวันตลอดเดือนกุมภาพันธ์ พ.ศ. 2568 ซึ่งแสดงผลเป็นรายสัปดาห์ (4 รอบ) เราสามารถสรุปพฤติกรรมการเดินทางได้ดังนี้:

1. **พฤติกรรมฟันปลา (Sawtooth Pattern):** ทุกสายรถไฟฟ้ามีรูปแบบการเดินทางที่ชัดเจนคือ ปริมาณผู้โดยสารจะพุ่งสูงที่สุดในวันทำงาน (จันทร์-ศุกร์) และตกลงอย่างรุนแรงในวันหยุดสุดสัปดาห์ (เสาร์-อาทิตย์) ซึ่งแสดงด้วยแถบไฮไลต์สีเทา
2. **การเปรียบเทียบระหว่างกลุ่มสายรถไฟฟ้า:**
   * **กลุ่มสายสีม่วง, ชมพู และเหลือง (Feeder Lines):** มีความอ่อนไหวต่อวันหยุดสูงมาก โดยปริมาณผู้โดยสารในวันเสาร์-อาทิตย์ หดตัวลงไปเหลือเพียงประมาณ **40-50%** เมื่อเทียบกับวันพีก
   * **กลุ่มสายสีน้ำเงิน, BTS และ ARL (Core Lines):** แม้จะมีการลดลงในวันหยุด แต่ยังรักษาระดับผู้โดยสารได้ดีกว่า โดยคงอยู่ที่ระดับ **65-75%** ของวันพีก เนื่องจากเป็นเส้นทางหลักที่เชื่อมต่อย่านธุรกิจและแหล่งท่องเที่ยว
3. **ข้อสรุปเชิงพฤติกรรม:** ข้อมูลนี้สะท้อนว่าระบบรถไฟฟ้าในกรุงเทพฯ ถูกใช้งานเพื่อการเดินทางไปทำงาน (Commuting) เป็นวัตถุประสงค์หลัก ทำให้เกิดความผันผวนของความต้องการใช้งาน (Demand) ที่แตกต่างกันอย่างมากระหว่างวันธรรมดาและวันหยุด

### Challenge Question 3

In [ ]:
# 1. Prepare Daily Total Rail Data
df_daily_total = df_rail.groupby('วันที่')['ปริมาณ'].sum().reset_index().sort_values('วันที่')

# 2. Create Visualization (Trend only)
fig_trend = go.Figure()

fig_trend.add_trace(go.Scatter(
    x=df_daily_total['วันที่'],
    y=df_daily_total['ปริมาณ'],
    mode='lines',
    name='ยอดผู้โดยสารรายวัน',
    line=dict(color='#183884', width=2)
))

fig_trend.update_layout(
    title='<b>Challenge 3: แนวโน้มปริมาณการเดินทางรายวัน (พ.ศ. 2568 - 2569)</b><br><sup>แสดงยอดผู้โดยสารรวมทุกเครือข่ายรถไฟฟ้ารายวัน</sup>',
    xaxis_title='วันที่',
    yaxis_title='จำนวนผู้โดยสารรวม (คน)',
    plot_bgcolor='white',
    hovermode='x unified',
    margin=dict(t=80, b=40, l=40, r=40)
)

fig_trend.update_yaxes(showgrid=True, gridcolor='whitesmoke')
fig_trend.update_xaxes(showgrid=True, gridcolor='whitesmoke')

fig_trend.show()

#### ความผิดปกติที่ชัดเจน: การใช้แถบทางสถิติ (Statistical Band)

เพื่อตรวจหาค่าที่ผิดปกติอย่างรุนแรง เราได้กำหนด **แถบทางสถิติ** ครอบคลุมชุดข้อมูลทั้งหมด

**ตรรกะในการวิเคราะห์:**
เราใช้ค่า **เบี่ยงเบนมาตรฐาน (Standard Deviation - STD)** ในการวัดระดับความแปรผันตามธรรมชาติของผู้โดยสาร โดยกำหนดช่วง 'พื้นที่ปลอดภัย' (Safe Zone) รอบค่าเฉลี่ยของจุด Peak และ Dip
*   **ขอบเขตบน:** ค่าเฉลี่ย Peak ประจำสัปดาห์ + 1.5 STD
*   **ขอบเขตล่าง:** ค่าเฉลี่ย Dip ประจำสัปดาห์ - 1.5 STD

วิธีนี้จะช่วยตรวจจับวันที่ผู้โดยสารทะลักหรือหายไปอย่างผิดปกติ เช่น ช่วงเทศกาลใหญ่หรือเหตุการณ์ที่ไม่คาดคิด

In [ ]:
# 1. เตรียมข้อมูลรายวัน และ จัดกลุ่มตามสัปดาห์
df_total_rail = df_challenge2.groupby('วันที่')['ปริมาณ'].sum().reset_index()
df_total_rail = df_total_rail.sort_values('วันที่').reset_index(drop=True)
df_total_rail['YearWeek'] = df_total_rail['วันที่'].dt.isocalendar().year.astype(str) + '-' + df_total_rail['วันที่'].dt.isocalendar().week.astype(str)

# 2. หา Peak และ Dip ประจำสัปดาห์
weekly_stats = df_total_rail.groupby('YearWeek')['ปริมาณ'].agg(['max', 'min']).reset_index()
weekly_stats.rename(columns={'max': 'Weekly_Peak', 'min': 'Weekly_Dip'}, inplace=True)
df_detect = pd.merge(df_total_rail, weekly_stats, on='YearWeek')

# 3. [อัปเกรด!] คํานวณค่าเฉลี่ย (Mean) และ ค่าเบี่ยงเบนมาตรฐาน (STD) ของ Peak และ Dip
avg_peak = weekly_stats['Weekly_Peak'].mean()
std_peak = weekly_stats['Weekly_Peak'].std()

avg_dip = weekly_stats['Weekly_Dip'].mean()
std_dip = weekly_stats['Weekly_Dip'].std()

# 4. กำหนด Margin ด้วย Z-Multiplier (จำนวนเท่าของ STD)
# ตามหลักสถิติ (Empirical Rule) ค่า 1.5 - 2 STD จะครอบคลุมข้อมูลความน่าจะเป็นปกติได้ 86% - 95%
z_multiplier_upper = 1.5  # ยอมให้ Peak ทะลุค่าเฉลี่ยไปได้ 1.5 ช่วงตัว
z_multiplier_lower = 1.5  # ยอมให้ Dip ร่วงจากค่าเฉลี่ยไปได้ 1.5 ช่วงตัว

# สร้างสมการขอบเขต (Threshold) จากสถิติ
upper_threshold = avg_peak + (z_multiplier_upper * std_peak)
lower_threshold = avg_dip - (z_multiplier_lower * std_dip)

print(f"--- Data-Driven Thresholds ---")
print(f"Upper: {avg_peak:,.0f} + (1.5 * {std_peak:,.0f}) = {upper_threshold:,.0f} คน")
print(f"Lower: {avg_dip:,.0f} - (1.5 * {std_dip:,.0f}) = {lower_threshold:,.0f} คน")

# 5. หาจุด Anomaly
df_detect['Is_Anomaly'] = (df_detect['ปริมาณ'] < lower_threshold) | (df_detect['ปริมาณ'] > upper_threshold)
anomalies = df_detect[df_detect['Is_Anomaly']]

# ---------------------------------------------------------
# การวาดกราฟ (Plotting)
# ---------------------------------------------------------
fig3 = go.Figure()

# เส้นจริง
fig3.add_trace(go.Scatter(
    x=df_detect['วันที่'], y=df_detect['ปริมาณ'],
    mode='lines', name='ปริมาณผู้โดยสารรวม',
    line=dict(color='#1f77b4', width=2),
    hovertemplate='%{x|%d %b %Y}<br>ปริมาณ: %{y:,.0f}<extra></extra>'
))

# เส้น Lower Threshold (Average Dip - 1.5 STD)
fig3.add_trace(go.Scatter(
    x=[df_detect['วันที่'].min(), df_detect['วันที่'].max()], y=[lower_threshold, lower_threshold],
    mode='lines', name=f'Lower Limit (-{z_multiplier_lower} STD)',
    line=dict(color='orange', width=2, dash='dash')
))

# เส้น Upper Threshold (Average Peak + 1.5 STD)
fig3.add_trace(go.Scatter(
    x=[df_detect['วันที่'].min(), df_detect['วันที่'].max()], y=[upper_threshold, upper_threshold],
    mode='lines', name=f'Upper Limit (+{z_multiplier_upper} STD)',
    line=dict(color='green', width=2, dash='dash')
))

# จุด Anomaly
fig3.add_trace(go.Scatter(
    x=anomalies['วันที่'], y=anomalies['ปริมาณ'],
    mode='markers', name='Anomaly (ผิดปกติ)',
    marker=dict(
        color=np.where(anomalies['ปริมาณ'] > upper_threshold, '#006400', '#d62728'),
        size=8, line=dict(color='white', width=1)
    ),
    hovertemplate='<b>🚨 Anomaly</b><br>วันที่: %{x|%d %b %Y}<br>ปริมาณ: %{y:,.0f}<extra></extra>'
))

fig3.update_layout(
    title='<b>Challenge 3: ตรวจจับความผิดปกติด้วยเทคนิค Statistical Band</b><br><sup>ใช้ค่าเบี่ยงเบนมาตรฐาน (Standard Deviation) เพื่อสร้างกรอบยืดหยุ่นแทนการใช้เปอร์เซ็นต์ตายตัว</sup>',
    yaxis_title='ปริมาณผู้โดยสารรวม (คน)',
    plot_bgcolor='white', hovermode='x unified', margin=dict(t=80, b=40, l=40, r=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig3.update_yaxes(showgrid=True, gridcolor='LightGray')
fig3.update_xaxes(showgrid=False, dtick="M1", tickformat="%b %Y")

fig3.show()

--- Data-Driven Thresholds ---
Upper: 1,685,339 + (1.5 * 152,316) = 1,913,814 คน
Lower: 966,401 - (1.5 * 140,745) = 755,283 คน


#### ความผิดปกติที่ซ่อนอยู่: การใช้ SciPy Peaks และโซนปลอดภัยแบบ STD

การใช้ขอบเขตคงที่มักพลาด 'ความผิดปกติที่ซ่อนอยู่' เช่น วันจันทร์ที่เงียบเหมือนวันอาทิตย์ หรือวันเสาร์ที่คนแน่นเหมือนวันจันทร์ ซึ่งค่าเหล่านี้อาจไม่เกินขอบเขตโดยรวม แต่ผิดปกติเมื่อเทียบกับประเภทของวันนั้นๆ

**ตรรกะ 'การเปลี่ยนแปลงของความชัน':**
เรานิยามจุด Peak และ Dip ว่าเป็นจุดที่ทิศทางของปริมาณผู้โดยสารเปลี่ยนไปอย่างกะทันหัน โดยใช้ฟังก์ชัน `find_peaks` จาก SciPy เพื่อแยกจุดเหล่านี้ออกมา

**ขั้นตอนการทำงาน:**
1.  **แยก Peak & Dip:** ค้นหาจุดสูงสุด (วันทำงาน) และจุดต่ำสุด (วันหยุด) ในแต่ละรอบสัปดาห์
2.  **กำหนดโซนปลอดภัยเฉพาะกลุ่ม:** คำนวณค่าเฉลี่ยและ STD ของกลุ่ม Peak และกลุ่ม Dip แยกออกจากกัน
3.  **ระบุข้อยกเว้น:**
    *   **Anomaly Peak:** คือวันทำงานที่มียอดผู้โดยสารหลุดจากช่วงปกติของวันทำงานด้วยกัน
    *   **Anomaly Dip:** คือวันหยุดที่มียอดผู้โดยสารหลุดจากช่วงปกติของวันหยุดด้วยกัน

วิธีนี้ช่วยให้เราตรวจพบวันหยุดที่คนเยอะผิดปกติได้ แม้จำนวนคนจะยังน้อยกว่าวันจันทร์ก็ตาม

In [ ]:
# ---------------------------------------------------------
# 1. Prepare data
# ---------------------------------------------------------
df_total_rail = df_challenge2.groupby('วันที่')['ปริมาณ'].sum().reset_index()
df_total_rail = df_total_rail.sort_values('วันที่').reset_index(drop=True)

y_values = df_total_rail['ปริมาณ'].values

# ---------------------------------------------------------
# 2. Find peaks & dips using SciPy (from snippet 1)
# ---------------------------------------------------------
peaks_idx, _ = find_peaks(y_values, distance=3, prominence=100000)
dips_idx, _  = find_peaks(-y_values, distance=3, prominence=100000)

all_peaks = df_total_rail.iloc[peaks_idx].copy()
all_dips  = df_total_rail.iloc[dips_idx].copy()

# ---------------------------------------------------------
# 3. Compute safe zones: avg ± 1 STD for peaks and dips
# ---------------------------------------------------------
avg_peak = all_peaks['ปริมาณ'].mean()
std_peak = all_peaks['ปริมาณ'].std()
avg_dip  = all_dips['ปริมาณ'].mean()
std_dip  = all_dips['ปริมาณ'].std()

peak_upper = avg_peak + (1.0 * std_peak)
peak_lower = avg_peak - (1.0 * std_peak)

dip_upper  = avg_dip + (1.0 * std_dip)
dip_lower  = avg_dip - (1.0 * std_dip)

# ---------------------------------------------------------
# 4. Flag anomalies — peaks vs peak zone, dips vs dip zone
# ---------------------------------------------------------
all_peaks['Is_Anomaly'] = (
    (all_peaks['ปริมาณ'] > peak_upper) | (all_peaks['ปริมาณ'] < peak_lower)
)
all_dips['Is_Anomaly'] = (
    (all_dips['ปริมาณ'] > dip_upper) | (all_dips['ปริมาณ'] < dip_lower)
)

anomaly_peaks = all_peaks[all_peaks['Is_Anomaly']]
anomaly_dips  = all_dips[all_dips['Is_Anomaly']]

# ---------------------------------------------------------
# 5. Plot
# ---------------------------------------------------------
fig = go.Figure()

# Raw line
fig.add_trace(go.Scatter(
    x=df_total_rail['วันที่'], y=df_total_rail['ปริมาณ'],
    mode='lines', name='ปริมาณผู้โดยสารรวม',
    line=dict(color='#1f77b4', width=2),
    hovertemplate='%{x|%d %b %Y}<br>ปริมาณ: %{y:,.0f}<extra></extra>'
))

# Safe zones
fig.add_hrect(y0=peak_lower, y1=peak_upper, fillcolor='green',  line_width=0, opacity=0.12)
fig.add_hrect(y0=dip_lower,  y1=dip_upper,  fillcolor='orange', line_width=0, opacity=0.12)

# Anomaly peaks
fig.add_trace(go.Scatter(
    x=anomaly_peaks['วันที่'], y=anomaly_peaks['ปริมาณ'],
    mode='markers', name='Anomaly Peak',
    marker=dict(color='#2ca02c', size=10, symbol='triangle-up',
                line=dict(color='darkgreen', width=1)),
    hovertemplate='<b>Anomaly Peak</b><br>%{x|%d %b %Y}<br>%{y:,.0f}<extra></extra>'
))

# Anomaly dips
fig.add_trace(go.Scatter(
    x=anomaly_dips['วันที่'], y=anomaly_dips['ปริมาณ'],
    mode='markers', name='Anomaly Dip',
    marker=dict(color='red', size=10, symbol='triangle-down',
                line=dict(color='darkred', width=1)),
    hovertemplate='<b>Anomaly Dip</b><br>%{x|%d %b %Y}<br>%{y:,.0f}<extra></extra>'
))

# ย้ายข้อความไปไว้ที่ Legend/Annotations ด้านบนแทน
fig.update_layout(
    title='<b>Anomaly Detection: SciPy Peaks + STD Safe Zones</b>',
    yaxis_title='ปริมาณผู้โดยสารรวม (คน)',
    plot_bgcolor='white', hovermode='x unified',
    margin=dict(t=120, b=40, l=40, r=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    annotations=[
        dict(x=0.01, y=1.08, xref="paper", yref="paper", text="🟩 <b>Peak Zone:</b> Avg Peak ± 1.0 STD", showarrow=False, font=dict(color="green")),
        dict(x=0.01, y=1.03, xref="paper", yref="paper", text="🟧 <b>Dip Zone:</b> Avg Dip ± 1.0 STD", showarrow=False, font=dict(color="orange"))
    ]
)
fig.update_yaxes(showgrid=True, gridcolor='LightGray')
fig.update_xaxes(showgrid=False, dtick="M1", tickformat="%b %Y")

fig.show()

#### บทวิเคราะห์ Challenge 3: การตรวจจับความผิดปกติขั้นสูง

**1. เหตุผลในการเลือกใช้กราฟ**
*   **กราฟเส้นพร้อมแถบสถิติ (Statistical Band):** ช่วยให้เห็นภาพรวมและขอบเขต 'พื้นที่ปลอดภัย' (Z-score > 1.5) ที่ครอบคลุมทั้งระบบ ทำให้ระบุวันที่เกิดเหตุการณ์กระทบต่อการเดินทางในวงกว้างได้ชัดเจน
*   **กราฟตรวจจับ Peak/Dip ด้วย SciPy:** กราฟนี้แยกการวิเคราะห์วันทำงาน (สีเขียว) และวันหยุด (สีส้ม) ออกจากกัน ซึ่งสำคัญมากเพราะปริมาณที่ 'ปกติ' ของวันอาทิตย์ อาจเป็นความ 'ผิดปกติ' ของวันจันทร์ได้ กราฟนี้จึงช่วยให้เราตรวจจับความเปลี่ยนแปลงที่ซ่อนอยู่ตามบริบทของวันนั้นๆ

**2. ข้อค้นพบสำคัญ**
*   **อิทธิพลจากปัจจัยภายนอก:** โมเดลตรวจพบ 'Triple Effect' (31 ม.ค. 2568) ที่เกิดจากเงินเดือนออก, เทศกาลตรุษจีน และมาตรการกระตุ้นเศรษฐกิจ ซึ่งสร้างยอดพีกสูงสุดในชุดข้อมูล
*   **การตอบสนองต่อวิกฤต:** พบ 'ยอดดิ่งฉุกเฉิน' ในวันที่ 28 มี.ค. 2568 (จากเหตุแผ่นดินไหว) ซึ่งพิสูจน์ว่าระบบสามารถตรวจจับพฤติกรรมประชาชนที่หลีกเลี่ยงการใช้รถไฟฟ้าในช่วงเกิดเหตุภัยธรรมชาติได้
*   **ปรากฏการณ์วันหยุดฟันหลอ:** ตรวจพบความผิดปกติในวันศุกร์หรือวันอังคารที่เชื่อมกับวันหยุดยาว สะท้อนว่าปริมาณผู้โดยสารผูกติดกับพนักงานออฟฟิศอย่างมาก เมื่อมีการลางานต่อเนื่อง ระบบรางจะรับรู้ถึงความต้องการที่ลดลงทันที

**3. มุมมองเชิงกลยุทธ์**
แม้รูปแบบฟันปลาจะเป็น 'ชีพจร' ปกติของกรุงเทพฯ แต่จุดที่ผิดปกติคือการตอบสนองต่อเศรษฐกิจ สิ่งแวดล้อม และเหตุฉุกเฉิน ข้อมูลนี้ช่วยให้ผู้ให้บริการปรับทรัพยากรให้เหมาะสม ไม่ใช่แค่ 'วันธรรมดา vs วันหยุด' แต่รวมถึงการเตรียมพร้อมสำหรับวันที่มีความเสี่ยงสูงด้วย

#### Dates Analysis


##### 🟢 1. Peak with High Anomaly (ล้นทะลักวันทำงาน)
*(Daily peaks significantly higher than average)*

กลุ่มนี้สะท้อนช่วงเวลาที่คนหลั่งไหลเข้ามาใช้ระบบรางหนาแน่นกว่าปกติมาก จากปัจจัยพิเศษ:

* **2025-01-31 (Friday): Triple Effect (เงินหมื่น + ตรุษจีน + สิ้นเดือน)**
    * **Context:** วันศุกร์สิ้นเดือน, หลังวันตรุษจีน (29 ม.ค.), และอยู่ในสัปดาห์เดียวกับที่รัฐบาลแจกเงิน 10,000 บาท เฟส 2 (27 ม.ค.)
    * **Insight:** เม็ดเงินสะพัด การจับจ่ายคึกคัก ส่งผลโดยตรงให้คนเดินทางออกมาร่วมกิจกรรมทางเศรษฐกิจอย่างมหาศาล
* **2025-02-07 (Friday): The "Mode Shift" Effect (หนีวิกฤต PM 2.5)**
    * **Context:** วิกฤตฝุ่น PM 2.5 ระดับสีแดงจัดในกรุงเทพฯ
    * **Insight:** แม้มีมาตรการ WFH แต่ยอดผู้โดยสารกลับพุ่งทะลุเพดาน แสดงให้เห็นว่าคนกรุงเทพฯ ละทิ้งระบบขนส่งเปิด (รถเมล์, วินมอเตอร์ไซค์) แล้วหนีลงมาใช้ระบบรางที่เป็นระบบปิด (Safe Haven) แทน
* **2025-02-14 (Friday): Valentine's Day Rebound**
    * **Context:** วันศุกร์แห่งความรัก และเป็นวันทำงานที่คั่นกลาง (Bridge Day) ต่อจากวันหยุดมาฆบูชา (พฤหัส 13 ก.พ.)
    * **Insight:** ปกติวัน Bridge Day คนมักจะลางานและยอดต้องตก แต่พลังของการออกไปเฉลิมฉลองเดตในคืนวันศุกร์ เอาชนะเอฟเฟกต์วันหยุดฟันหลอได้อย่างราบคาบ

---

### 🔴 2. Peak with Low Anomaly (วันทำงานที่คนหายวับ)
*(Daily peaks significantly lower than average)*

กลุ่มนี้สะท้อนพฤติกรรม "วันหยุดฟันหลอ (Bridge Days)" และการหนีเมืองช่วงก่อน/หลังเทศกาล:

* **2025-01-03 (Friday):** วันหยุดฟันหลอหลังเทศกาลปีใหม่ (ลากยาวหยุด 5 วัน)
* **2025-04-04 (Friday):** วันหยุดฟันหลอก่อนวันจักรี (วันจักรีตรงกับอาทิตย์ 6 เม.ย., ชดเชยจันทร์ 7 เม.ย.) คนเริ่มออกต่างจังหวัด
* **2025-04-08 (Tuesday): The "Mega Bridge Week"**
    * **Context:** วันทำงานวันแรกหลังหยุดยาววันจักรี และเป็นโค้งสุดท้ายก่อนสงกรานต์
    * **Insight:** คนจำนวนมากใช้วันลาเพื่อเชื่อมวันหยุดจักรีเข้ากับสงกรานต์ สร้าง Long Weekend 10 กว่าวัน ส่งผลให้สัปดาห์นี้ยอดร่วงทั้งสัปดาห์
* **2025-04-18 (Friday):** วันหยุดฟันหลอหลังเทศกาลสงกรานต์
* **2025-05-06 (Tuesday):** วันหยุดฟันหลอหลังวันฉัตรมงคล (วันฉัตรมงคลตรงกับจันทร์ 5 พ.ค.)
* **2025-12-31 (Wednesday):** วันสิ้นปี (New Year's Eve) วันหยุดราชการกลางสัปดาห์
* **2026-03-02 (Monday):** วันหยุดฟันหลอก่อนวันมาฆบูชา (วันมาฆบูชาตรงกับอังคาร 3 มี.ค. 69)

---

### 🟡 3. Dip with High Anomaly (วันหยุด/วันก้นเหวที่คนแน่นผิดปกติ)
*(Weekend/Holiday lows that remained higher than typical lows)*

กลุ่มก้นเหวที่ไม่ลึกถึงเกณฑ์ปกติ สะท้อนถึงแรงจูงใจพิเศษที่ดึงคนออกจากบ้าน:

* **2025-01-16 (Thursday): เอฟเฟกต์วันครู**
    * **Context:** โรงเรียนหยุด แต่ผู้ใหญ่ยังทำงาน
    * **Insight:** ยอดเด็กลดลงจนกราฟตกลงมาเป็น Local Dip กลางสัปดาห์ แต่ไม่ได้ตกลึกเท่าวันเสาร์-อาทิตย์ เพราะมวลชนวัยทำงานยังพยุงยอดไว้
* **2025-01-26 (Sunday): วันจ่ายก่อนตรุษจีน**
    * **Context:** วันอาทิตย์สุดท้ายก่อนเริ่มเทศกาลตรุษจีน
    * **Insight:** ประชาชนออกไปจับจ่ายซื้อของไหว้ตามย่านการค้าหลัก ทำให้ยอดวันอาทิตย์นี้สูงผิดปกติ
* **2025-02-02 (Sunday): Post-Payday Weekend** (วันอาทิตย์แรกของเดือนหลังเงินเดือนออก)
* **2026-02-01 (Sunday): Post-Payday Weekend** (วันอาทิตย์แรกของเดือนหลังเงินเดือนออก)
* **2026-02-08 (Sunday): Kaset Fair / Mega Event Effect**
    * **Context:** วันอาทิตย์ที่สองของเดือนกุมภาพันธ์
    * **Insight:** เป็นช่วง High Season ของการจัดงานแฟร์ระดับประเทศในกรุงเทพฯ (เช่น งานเกษตรแฟร์) ซึ่งตรงกับวันหยุดพอดี ดึงดูดมวลชนให้แห่ไปใช้บริการรถไฟฟ้าจนล้นสถานี

---

### 🟤 4. Dip with Low Anomaly (วันหยุด/วันก้นเหวที่เมืองร้างแบบสุดๆ)
*(Weekend/Holiday lows that dropped significantly lower than average)*

กลุ่มนี้คือช่วงที่ประชาชนละทิ้งเมืองอย่างสมบูรณ์ หรือเกิดวิกฤตร้ายแรง:

* **2025-03-28 (Friday): วิกฤตแผ่นดินไหว & อาคารถล่ม (Emergency Drop)**
    * **Context:** เหตุแผ่นดินไหวเมียนมาสะเทือนถึงไทย ทำให้อาคารสูงพังถล่มใน กทม.
    * **Insight:** วันศุกร์ที่ควรจะเป็น Peak กลับร่วงลงมาเป็นก้นเหวลึก (Dip Low Anomaly) สะท้อนความตื่นตระหนก การประกาศพื้นที่ฉุกเฉิน และการหลีกเลี่ยงระบบขนส่งสาธารณะโดยฉับพลัน
* **2025-05-12 (Monday):** วันหยุดชดเชยวันวิสาขบูชา (หยุดยาว 3 วัน คนทิ้งกรุงเทพฯ)
* **2025-06-03 (Tuesday):** วันเฉลิมพระชนมพรรษา สมเด็จพระนางเจ้าฯ พระบรมราชินี
* **2025-07-28 (Monday):** วันเฉลิมพระชนมพรรษา พระบาทสมเด็จพระเจ้าอยู่หัว (Long Weekend)
* **2025-12-10 (Wednesday):** วันรัฐธรรมนูญ (วันหยุดกลางสัปดาห์ที่คนเลือกพักผ่อนอยู่บ้าน)



## 5.Dashboard Preparation


ในส่วนสุดท้ายนี้ เราจะทำการรวบรวมผลลัพธ์จากการวิเคราะห์ทั้งหมดมาสรุปไว้ในรูปแบบของ **Executive Dashboard** เพื่อให้เห็นภาพรวมของระบบขนส่งทางรางได้อย่างครบถ้วนในที่เดียว

**สิ่งที่ดำเนินการในขั้นตอนนี้:**
1.  **การคำนวณตัวชี้วัดหลัก (Core KPIs):** สรุปยอดผู้โดยสารรวม, ค่าเฉลี่ยการเดินทางรายวัน และอัตราการเติบโตแบบ YTD เพื่อเป็นหัวใจสำคัญของรายงาน
2.  **การรวมผลลัพธ์จาก Challenge ต่างๆ:** นำกราฟที่สร้างไว้ก่อนหน้า ทั้งเรื่องส่วนแบ่งตลาด (Market Share), การเติบโต (Growth), การกระจายตัวของข้อมูล (Distribution), ดัชนีความผันผวน (Volatility Index) และการตรวจจับความผิดปกติ (Anomaly Detection) มาจัดวางอย่างเป็นระบบ
3.  **การวิเคราะห์ในมุมมองผู้บริหาร:** เน้นการใช้สีสื่อความหมาย (Color Coding) และการเปรียบเทียบเชิงสถิติเพื่อให้ข้อมูลมีความหมายและสามารถนำไปใช้ในการตัดสินใจเชิงกลยุทธ์ได้ทันที

In [ ]:
# 1. Calculate Core KPIs for Dashboard
total_passengers = df_rail['ปริมาณ'].sum()
avg_daily_passengers = df_rail.groupby('วันที่')['ปริมาณ'].sum().mean()

# Calculate YTD Growth for Dashboard Header
ytd_stats = df_ytd.groupby('ปี')['ปริมาณ'].sum()
growth_rate = ((ytd_stats[2026] - ytd_stats[2025]) / ytd_stats[2025]) * 100

print(f"Dashboard KPIs Ready:")
print(f"- Total Rail Passengers: {total_passengers:,.0f}")
print(f"- Avg Daily Volume: {avg_daily_passengers:,.0f}")
print(f"- YTD Growth Rate: {growth_rate:+.2f}%")

Dashboard KPIs Ready:
- Total Rail Passengers: 620,232,351
- Avg Daily Volume: 1,425,821
- YTD Growth Rate: -5.82%


In [ ]:
# --- Re-prepare required data for the dashboard ---
agg_net = df_rail.groupby('กลุ่มรถไฟฟ้า')['ปริมาณ'].sum().sort_values(ascending=False).reset_index()

# Define color maps
rail_group_colors = {
    'BTS': '#42b944',
    'MRT': '#183884',
    'Airport Rail Link': '#8a2132',
    'SRT Red Line': '#d3273e'
}

rail_line_colors = {
    'BTS': '#42b944', 'MRT Blue': '#183884', 'MRT Purple': '#800080',
    'MRT Yellow': '#ffcc00', 'MRT Pink': '#ff66cc', 'ARL': '#8a2132', 'SRT Red': '#d3273e'
}

# Map for exact target_lines matching
target_line_color_map = {
    'รถไฟฟ้า BTS': '#42b944',
    'รถไฟฟ้าสายสีน้ำเงิน': '#183884',
    'รถไฟฟ้าสายสีม่วง': '#800080',
    'รถไฟฟ้าสายสีเหลือง': '#ffcc00',
    'รถไฟฟ้าสายสีชมพู': '#ff66cc',
    'รถไฟฟ้า ARL': '#8a2132',
    'รถไฟฟ้าสายสีแดง': '#d3273e'
}

pane1_colors = [rail_group_colors.get(name, '#183884') for name in agg_net['กลุ่มรถไฟฟ้า']]
pane5_colors = [rail_line_colors.get(name, '#bdc3c7') for name in stats['สายรถไฟฟ้า']]
growth_colors = ['#2ca02c' if x > 0 else '#d62728' for x in df_pivot['Growth (%)']]

fig_dash = make_subplots(
    rows=3, cols=2,
    subplot_titles=("<b>1. Total Cumulative Volume</b>",
                    "<b>2. YTD Growth (2568 vs 2569)</b>",
                    "<b>3. Daily Volume Distribution</b>",
                    "<b>4. Market Share % over Time</b>",
                    "<b>5. Network Volatility Index (CV%)</b>",
                    "<b>6. System-wide Anomaly Detection</b>"),
    vertical_spacing=0.1,
    horizontal_spacing=0.1
)

# Pane 1: Total Volume
fig_dash.add_trace(go.Bar(x=agg_net['กลุ่มรถไฟฟ้า'], y=agg_net['ปริมาณ'],
                         marker_color=pane1_colors, name='Total Volume'), row=1, col=1)

# Pane 2: YTD Growth
fig_dash.add_trace(go.Bar(x=df_pivot['Growth (%)'], y=df_pivot['กลุ่มรถไฟฟ้า'],
                         orientation='h', marker_color=growth_colors, name='Growth %'), row=1, col=2)

# Pane 3: Distribution (Fixed Colors)
for line in target_lines:
    subset = df_challenge2[df_challenge2['ยานพาหนะ/ท่า'] == line]
    clean_name = line.replace('รถไฟฟ้า', '').strip()
    box_color = target_line_color_map.get(line, '#bdc3c7')
    fig_dash.add_trace(go.Box(y=subset['ปริมาณ'], name=clean_name,
                             marker_color=box_color, boxpoints=False), row=2, col=1)

# Pane 4: Market Share (Fixed Order: Large to Small)
area_order = ['BTS', 'MRT', 'Airport Rail Link', 'SRT Red Line']
for col in area_order:
    if col in df_perc.columns:
        area_color = rail_group_colors.get(col, '#bdc3c7')
        fig_dash.add_trace(go.Scatter(x=df_perc.index, y=df_perc[col], stackgroup='one',
                                     name=col, fill='tonexty', line=dict(color=area_color)), row=2, col=2)

# Pane 5: Volatility Index
fig_dash.add_trace(go.Bar(x=stats['CV (%)'], y=stats['สายรถไฟฟ้า'],
                         orientation='h', marker_color=pane5_colors, name='Volatility'), row=3, col=1)

# Pane 6: Anomaly Detection
fig_dash.add_trace(go.Scatter(x=df_total_rail['วันที่'], y=df_total_rail['ปริมาณ'],
                             mode='lines', line=dict(color='#bdc3c7', width=1), name='Trend'), row=3, col=2)
fig_dash.add_trace(go.Scatter(x=anomaly_peaks['วันที่'], y=anomaly_peaks['ปริมาณ'],
                             mode='markers', marker=dict(color='green', size=6, symbol='triangle-up'), name='High'), row=3, col=2)
fig_dash.add_trace(go.Scatter(x=anomaly_dips['วันที่'], y=anomaly_dips['ปริมาณ'],
                             mode='markers', marker=dict(color='red', size=6, symbol='triangle-down'), name='Low'), row=3, col=2)

fig_dash.update_layout(height=1400, title_text=f"<b>Rail Connectivity Executive Command Center</b><br><sup>Total Passengers: {total_passengers:,.0f} | YTD Growth: {growth_rate:+.2f}%</sup>",
                  showlegend=False, plot_bgcolor='white')
fig_dash.update_xaxes(showgrid=True, gridcolor='whitesmoke')
fig_dash.update_yaxes(showgrid=True, gridcolor='whitesmoke')
fig_dash.show()

## 5. สรุป Key Insights ที่ค้นพบ (Executive Summary)

จากการเดินทางของข้อมูลตั้งแต่การทำความสะอาดไปจนถึงการสร้าง Dashboard ขั้นสูง เราสามารถสรุปข้อค้นพบสำคัญได้ 3 ประเด็นหลักดังนี้:

### 🟢 **1. โครงสร้างตลาดและการเติบโต (Market Dynamics & Growth)**
*   **ผู้นำตลาด:** **BTS และ MRT** ยังคงเป็นกระดูกสันหลังหลักของกรุงเทพฯ โดยครองส่วนแบ่งตลาดรวมกันมากกว่า **90%** อย่างต่อเนื่อง
*   **สัญญาณการขยายตัว:** แม้ภาพรวมปี 2569 จะดูทรงตัว แต่เราพบการเติบโตที่น่าสนใจในสาย **SRT Red Line และ Airport Rail Link** ในช่วงต้นปี ซึ่งสะท้อนถึงการปรับเปลี่ยนพฤติกรรมของผู้ที่อยู่อาศัยแถบชานเมืองและการฟื้นตัวของภาคการท่องเที่ยว

### 🟡 **2. พฤติกรรมและความเสถียร (Passenger Behavior & Stability)**
*   **ความต่างของประเภทเส้นทาง:**
    *   **กลุ่ม Feeder (สายสีม่วง, เหลือง, ชมพู):** มีความผันผวนสูงมาก โดยยอดผู้โดยสารจะหายไปกว่า **40-45% ในวันหยุด (Weekend Drop)** เนื่องจากผู้ใช้หลักคือกลุ่มคนทำงาน (Commuters)
    *   **กลุ่ม Core (BTS, MRT Blue, ARL):** มีความเสถียรสูงกว่า โดยรักษาระดับผู้โดยสารในวันหยุดไว้ได้ที่ **65-75%** ของวันธรรมดา
*   **จุดเด่นของ ARL:** เป็นสายที่มีความเสถียรสูงสุดเนื่องจากตอบโจทย์ทั้งการทำงานและการเชื่อมต่อสนามบิน

### 🔴 **3. การตรวจจับสิ่งผิดปกติ (Advanced Anomaly Insights)**
*   **ปัจจัยภายนอกที่มีผลรุนแรง:** อัลกอริทึมตรวจพบว่าเหตุการณ์พิเศษมีผลต่อยอดผู้โดยสารอย่างชัดเจน เช่น **มาตรการกระตุ้นเศรษฐกิจ (เงินหมื่น)** และ **วิกฤต PM 2.5** ที่ดึงคนจากถนนลงสู่ระบบรางที่เป็นระบบปิด
*   **การตอบสนองต่อเหตุการณ์ฉุกเฉิน:** ระบบสามารถตรวจจับ **'ยอดดิ่งฉุกเฉิน'** ในช่วงเหตุการณ์แผ่นดินไหว (มีนาคม 2568) ซึ่งพิสูจน์ว่าประชาชนจะหลีกเลี่ยงการใช้รถไฟฟ้าทันทีเมื่อมีความกังวลด้านความปลอดภัย
*   **โอกาสในการบริหารจัดการ:** การพบ **'วันทำงานที่คนหาย' (Bridge Days)** ช่วยให้ผู้ให้บริการสามารถวางแผนลดรอบรถเพื่อประหยัดต้นทุนในวันที่คนส่วนใหญ่ลางานต่อเนื่องได้

**บทสรุป:** ข้อมูลนี้สะท้อนว่าระบบรางในกรุงเทพฯ คือ 'เครื่องวัดชีพจร' ของเมืองที่ตอบสนองต่อเศรษฐกิจ สังคม และสิ่งแวดล้อมอย่างรวดเร็ว การเข้าใจความผันผวนเหล่านี้จะช่วยให้การวางแผนเชิงกลยุทธ์ในอนาคตมีความแม่นยำยิ่งขึ้น